# 🚀 00 — Environment Setup
## Ekegusii-LLM-Translation · Kineses Cloud / Base Jupyter

**Run this notebook FIRST before opening any other notebook.**

This notebook:
1. Installs ALL required packages into the **exact Python kernel** running in Jupyter (fixes `ModuleNotFoundError`).
2. Downloads & extracts the repository ZIP directly via Python — **no `git` binary needed**.
3. Verifies the Master Corpus data integrity (0% leakage guarantee).
4. Confirms GPU availability and hardware specs.

---
| Node Spec | Value |
|-----------|-------|
| CPU Cores | 22 |
| RAM | 117.9 GB |
| Disk | 967.64 GB |
| Target GPU | NVIDIA A100 80GB |

In [ ]:
# ============================================================
# CELL 1: FORCE INSTALL ALL PACKAGES INTO THE CORRECT KERNEL
# ============================================================
# Uses sys.executable to guarantee packages are installed into
# the EXACT same Python binary that this notebook is running on.
# This permanently fixes: ModuleNotFoundError: No module named 'pandas'
# ============================================================
import sys
import subprocess

def pip_install(packages):
    """Install packages into the EXACT Python kernel that is running."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + packages
    )

print(f"Active Python Kernel  : {sys.executable}")
print(f"Python Version        : {sys.version}")
print("\nInstalling packages into this kernel...\n")

# --- Core Data Science ---
print("[1/6] Installing core data science packages...")
pip_install(["pandas", "numpy", "matplotlib", "seaborn", "plotly", "scipy", "scikit-learn"])

# --- HuggingFace Ecosystem ---
print("[2/6] Installing HuggingFace ecosystem...")
pip_install(["transformers", "datasets", "evaluate", "tokenizers", "accelerate", "huggingface_hub"])

# --- QLoRA Fine-Tuning Stack ---
print("[3/6] Installing QLoRA stack (PEFT + TRL + BitsAndBytes)...")
pip_install(["peft", "trl", "bitsandbytes"])

# --- Evaluation Metrics ---
print("[4/6] Installing evaluation metric libraries...")
pip_install(["sacrebleu", "unbabel-comet", "nltk"])

# --- CLI & Config Tools ---
print("[5/6] Installing CLI and config tools...")
pip_install(["rich", "typer", "hydra-core", "omegaconf"])

# --- Utilities ---
print("[6/6] Installing utility packages...")
pip_install(["tqdm", "ipywidgets", "jupyterlab"])

print("\n" + "="*60)
print("✅ ALL PACKAGES INSTALLED INTO CORRECT KERNEL!")
print("="*60)
print("\nVerifying imports...")

import torch
import pandas as pd
import numpy as np
import transformers
import peft
import sacrebleu

print(f"\n{'='*60}")
print(f"  PyTorch Version    : {torch.__version__}")
print(f"  Pandas Version     : {pd.__version__}")
print(f"  NumPy Version      : {np.__version__}")
print(f"  Transformers       : {transformers.__version__}")
print(f"  PEFT Version       : {peft.__version__}")
print(f"  SacreBLEU Version  : {sacrebleu.__version__}")
print(f"  CUDA Available     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU Name           : {torch.cuda.get_device_name(0)}")
    print(f"  GPU VRAM           : {props.total_memory / 1e9:.1f} GB")
    print(f"  GPU Compute Cap.   : {props.major}.{props.minor}")
else:
    print("  ⚠️  No GPU detected. Training will run on CPU only.")

print(f"{'='*60}")
print("✅ ENVIRONMENT VERIFIED & READY!")

In [ ]:
# ============================================================
# CELL 2: DOWNLOAD & SETUP REPOSITORY (NO GIT NEEDED)
# ============================================================
# Downloads the full research repository as a ZIP directly
# from GitHub using only Python standard library modules.
# No git binary, no terminal, no admin permissions needed.
# ============================================================
import urllib.request
import zipfile
import os
import sys

REPO_URL = "https://github.com/aykahsay/Ekegusii-LLM-Translation/archive/refs/heads/main.zip"
ZIP_NAME = "repo.zip"
PROJ_DIR = "Ekegusii-LLM-Translation-main"

if not os.path.isdir(PROJ_DIR):
    print("📥 Downloading repository ZIP (~15 MB)...")
    urllib.request.urlretrieve(REPO_URL, ZIP_NAME)
    print("📦 Extracting files...")
    with zipfile.ZipFile(ZIP_NAME, "r") as z:
        z.extractall(".")
    os.remove(ZIP_NAME)
    print(f"✅ Repository extracted into: {PROJ_DIR}/")
else:
    print(f"✅ Repository already present at: {PROJ_DIR}/")
    print("   Skipping download.")

# --- Set Working Directory & sys.path ---
if os.path.basename(os.getcwd()) != PROJ_DIR:
    os.chdir(PROJ_DIR)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"\n📁 Working Directory : {os.getcwd()}")
print(f"🐍 Python Kernel     : {sys.executable}")

# --- Verify Expected Folder Structure ---
print("\n📂 Verifying repository structure:")
expected = [
    ("src",                            "Core Python modules"),
    ("data",                           "Master corpora and splits"),
    ("notebooks",                      "13 research notebooks"),
    ("configs",                        "Hydra YAML configuration files"),
    ("tests",                          "Unit test suite"),
    ("scripts",                        "Training pipeline shell scripts"),
    ("data/master_corpus",             "Master corpus CSV files"),
    ("src/master_corpus",              "MasterCorpusManager module"),
    ("src/task_generation",           "InstructionTaskGenerator module"),
    ("src/evaluation",                "Evaluation metric modules"),
    ("src/models",                    "QLoRA model loading modules"),
]
all_ok = True
for folder, desc in expected:
    exists = os.path.isdir(folder)
    status = "✅" if exists else "❌ MISSING"
    if not exists:
        all_ok = False
    print(f"  {status}  {folder:<35} — {desc}")

if all_ok:
    print("\n✅ FULL REPOSITORY STRUCTURE VERIFIED!")
else:
    print("\n⚠️  Some directories are missing. Check the GitHub repository.")

In [ ]:
# ============================================================
# CELL 3: VERIFY MASTER CORPUS & 0% DATA LEAKAGE GUARANTEE
# ============================================================
from src.master_corpus.manager import MasterCorpusManager
from src.master_corpus.integrity import DataLeakageChecker

print("Loading Master Corpus...")
manager = MasterCorpusManager()

corpus = manager.load_sentence_corpus()
lexical = manager.load_lexical_corpus()
train  = manager.load_train_split()
val    = manager.load_val_split()
test   = manager.load_test_split()

print(f"\n{'='*60}")
print(f"  Master Sentence Corpus : {len(corpus):,} multilingual concepts")
print(f"  Master Lexical Corpus  : {len(lexical):,} dictionary entries")
print(f"  Train Split            : {len(train):,} concepts  (80%)")
print(f"  Val   Split            : {len(val):,} concepts  (10%)")
print(f"  Test  Split            : {len(test):,} concepts  (10%)")
print(f"{'='*60}")

print("\nRunning Data Leakage Audit...")
checker = DataLeakageChecker(manager)
checker.verify_all()
print("✅ 0% DATA LEAKAGE CONFIRMED ACROSS ALL SPLITS!")
print("\n🎉 SETUP COMPLETE — You may now open any notebook in notebooks/")

print("\n📋 NEXT STEPS:")
print("  → notebooks/05_instruction_generation.ipynb  (Generate 6-Way Tasks)")
print("  → notebooks/07_train_aya.ipynb               (Train Cohere Aya-23 8B)")
print("  → notebooks/08_train_llama.ipynb             (Train Llama-3.1 8B)")
print("  → notebooks/09_translation_evaluation.ipynb  (SacreBLEU / chrF++ / COMET)")